In [ ]:
from __future__ import annotations

import ast
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT_DIR = Path(r"C:\Users\mathlouthi\Desktop\POSS-LOGIC-LM-GITHUB")

POSS_RESULTS_DIR = PROJECT_DIR / "POSS-LOGICLM"

OUTPUT_DIR = PROJECT_DIR / "analysis_outputs" / "final_figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_PNG = OUTPUT_DIR / "figure5_necessity_analysis.png"
FIGURE_PDF = OUTPUT_DIR / "figure5_necessity_analysis.pdf"
SUMMARY_CSV = OUTPUT_DIR / "figure5_necessity_analysis_summary.csv"
SOURCE_CSV = OUTPUT_DIR / "figure5_source_files.csv"
DIAGNOSTIC_CSV = OUTPUT_DIR / "figure5_diagnostic_files.csv"


# ============================================================
# 2. ORDRES ET STYLE
# ============================================================

DATASET_ORDER = ["PrOntoQA", "ProofWriter", "FOLIO"]
MODEL_ORDER = ["Gemma", "LLaMA", "Qwen"]

CATEGORY_ORDER = [
    "TF consensus",
    "Semantic Unknown",
    "Conflict",
    "Technical Unknown",
]

CATEGORY_COLORS = {
    "TF consensus": "#5B9BD5",
    "Semantic Unknown": "#3CB371",
    "Conflict": "#E07B52",
    "Technical Unknown": "#A6A6A0",
}

CATEGORY_LABELS = {
    "TF consensus": r"$N=1$ T/F consensus",
    "Semantic Unknown": r"$N=1$ semantic Unknown",
    "Conflict": r"$N<1$ conflict",
    "Technical Unknown": "Technical Unknown",
}

MIN_INTERNAL_LABEL_PCT = 7.0
MIN_EXTERNAL_LABEL_PCT = 1.0


# ============================================================
# 3. ALIASES POUR RECONNAÎTRE LES FICHIERS
# ============================================================

DATASET_ALIASES = {
    "PrOntoQA": ["prontoqa", "pqa"],
    "ProofWriter": ["proofwriter", "pw"],
    "FOLIO": ["folio"],
}

MODEL_ALIASES = {
    "Gemma": ["gemma"],
    "LLaMA": ["llama", "llama3", "llama31", "llama_3"],
    "Qwen": ["qwen"],
}


# ============================================================
# 4. HELPERS TEXTE
# ============================================================

def normalize_text(value: Any) -> str:
    text = str(value).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return f" {text.strip()} "


def contains_alias(text: str, aliases: List[str]) -> bool:
    text_norm = normalize_text(text)

    for alias in aliases:
        alias_norm = normalize_text(alias).strip()

        if f" {alias_norm} " in text_norm:
            return True

        if alias_norm in text_norm:
            return True

    return False


def path_text(path: Path) -> str:
    return str(path).replace("\\", "/")


# ============================================================
# 5. LECTURE JSON
# ============================================================

def read_json(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_results(data: Dict[str, Any]) -> List[Dict[str, Any]]:
    for key in ["results", "data", "items", "examples", "records"]:
        value = data.get(key)

        if isinstance(value, list):
            return [row for row in value if isinstance(row, dict)]

    raise ValueError("Aucune liste results/data/items/examples/records trouvée dans le JSON.")


# ============================================================
# 6. RECHERCHE AUTOMATIQUE DES FICHIERS PBS / POSS
# ============================================================

def list_json_files(root: Path) -> List[Path]:
    if not root.exists():
        return []

    return sorted(root.rglob("*.json"))


def score_candidate(path: Path, dataset: str, model: str) -> int:
    full = path_text(path)
    name = path.name

    dataset_aliases = DATASET_ALIASES[dataset]
    model_aliases = MODEL_ALIASES[model]

    score = 0

    if contains_alias(name, dataset_aliases):
        score += 8
    elif contains_alias(full, dataset_aliases):
        score += 4

    if contains_alias(name, model_aliases):
        score += 8
    elif contains_alias(full, model_aliases):
        score += 5

    name_norm = normalize_text(name)
    full_norm = normalize_text(full)

    if " pbs " in name_norm or " pbs " in full_norm:
        score += 5

    if " results " in name_norm or " result " in name_norm:
        score += 2

    if " logiclm " in name_norm:
        score -= 4

    if " nosr " in name_norm:
        score -= 4

    if " greedy " in name_norm:
        score -= 4

    if " beam " in name_norm:
        score -= 3

    return score


def find_poss_file(dataset: str, model: str) -> Tuple[Optional[Path], List[Tuple[int, Path]]]:
    all_files = list_json_files(POSS_RESULTS_DIR)

    scored: List[Tuple[int, Path]] = []

    for path in all_files:
        score = score_candidate(path, dataset, model)

        if score > 0:
            scored.append((score, path))

    scored.sort(key=lambda item: item[0], reverse=True)

    strong_candidates = [(score, path) for score, path in scored if score >= 10]

    if strong_candidates:
        return strong_candidates[0][1], scored[:10]

    return None, scored[:10]


def build_diagnostic_report() -> pd.DataFrame:
    rows = []

    for dataset in DATASET_ORDER:
        for model in MODEL_ORDER:
            selected, candidates = find_poss_file(dataset, model)

            rows.append(
                {
                    "Dataset": dataset,
                    "Model": model,
                    "Root": str(POSS_RESULTS_DIR),
                    "Selected": str(selected) if selected else "",
                    "Status": "FOUND" if selected else "MISSING",
                    "Top candidates": " | ".join(
                        [f"{score}: {path}" for score, path in candidates[:5]]
                    ),
                }
            )

    return pd.DataFrame(rows)


# ============================================================
# 7. NORMALISATION DES SORTIES SOLVER
# ============================================================

def normalize_label(value: Any) -> str:
    text = str(value).strip().lower()

    true_values = {
        "true",
        "t",
        "yes",
        "entailment",
        "entailed",
        "proved",
        "valid",
    }

    false_values = {
        "false",
        "f",
        "no",
        "contradiction",
        "contradicted",
        "disproved",
        "invalid",
    }

    unknown_values = {
        "unknown",
        "unk",
        "uncertain",
        "none",
        "nan",
        "null",
        "",
        "not enough information",
        "not_enough_information",
    }

    if text in true_values:
        return "True"

    if text in false_values:
        return "False"

    if text in unknown_values:
        return "Unknown"

    if "true" in text:
        return "True"

    if "false" in text:
        return "False"

    if "unknown" in text or "uncertain" in text:
        return "Unknown"

    return "Unknown"


def parse_solver_outputs(value: Any) -> List[str]:
    if value is None:
        return []

    if isinstance(value, list):
        return [normalize_label(v) for v in value]

    if isinstance(value, tuple):
        return [normalize_label(v) for v in value]

    text = str(value).strip()

    try:
        parsed = ast.literal_eval(text)

        if isinstance(parsed, list):
            return [normalize_label(v) for v in parsed]

        if isinstance(parsed, tuple):
            return [normalize_label(v) for v in parsed]

    except Exception:
        pass

    labels = re.findall(
        r"True|False|Unknown|Uncertain|Entailment|Contradiction|PROVED|DISPROVED",
        text,
        flags=re.IGNORECASE,
    )

    return [normalize_label(v) for v in labels]


def get_solver_outputs(row: Dict[str, Any]) -> List[str]:
    candidate_keys = [
        "solver_out",
        "solver_outputs",
        "solver",
        "labels",
        "beam_labels",
        "candidate_predictions",
        "candidate_outputs",
        "predictions",
        "answers",
        "poss_labels",
        "necessity_labels",
    ]

    for key in candidate_keys:
        if key in row:
            outputs = parse_solver_outputs(row[key])

            if outputs:
                return outputs

    candidates = row.get("candidates")

    if isinstance(candidates, list):
        outputs = []

        for candidate in candidates:
            if not isinstance(candidate, dict):
                continue

            for key in ["solver_output", "solver_out", "prediction", "pred", "label", "answer"]:
                if key in candidate:
                    outputs.append(normalize_label(candidate.get(key)))
                    break

        if outputs:
            return outputs

    beam = row.get("beam")

    if isinstance(beam, list):
        outputs = []

        for candidate in beam:
            if not isinstance(candidate, dict):
                continue

            for key in ["solver_output", "solver_out", "prediction", "pred", "label", "answer"]:
                if key in candidate:
                    outputs.append(normalize_label(candidate.get(key)))
                    break

        if outputs:
            return outputs

    single_prediction_keys = [
        "pred",
        "prediction",
        "answer",
        "final_answer",
        "predicted_label",
        "final_prediction",
    ]

    for key in single_prediction_keys:
        if key in row:
            return [normalize_label(row[key])]

    return []


# ============================================================
# 8. CALCUL DU CORRECT
# ============================================================

def to_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value

    if isinstance(value, int):
        return value == 1

    if isinstance(value, float):
        return value == 1.0

    if isinstance(value, str):
        value_norm = value.strip().lower()
        return value_norm in {"true", "1", "yes", "correct", "ok"}

    return False


def row_is_correct(row: Dict[str, Any]) -> bool:
    correct_keys = [
        "correct",
        "is_correct",
        "success",
        "is_success",
        "prediction_correct",
    ]

    for key in correct_keys:
        if key in row:
            return to_bool(row[key])

    gold_keys = ["gold", "label", "target", "ground_truth", "answer"]
    pred_keys = ["pred", "prediction", "final_answer", "predicted_label", "final_prediction"]

    gold_value = None
    pred_value = None

    for key in gold_keys:
        if key in row:
            gold_value = row.get(key)
            break

    for key in pred_keys:
        if key in row:
            pred_value = row.get(key)
            break

    gold = normalize_label(gold_value)
    pred = normalize_label(pred_value)

    return gold == pred


# ============================================================
# 9. CLASSIFICATION DES INSTANCES
# ============================================================

def classify_instance(dataset: str, solver_outputs: List[str]) -> str:
    """
    Catégories Figure 5 :
    - TF consensus : tous les candidats donnent True ou tous donnent False.
    - Semantic Unknown : tous les candidats donnent Unknown sur PrOntoQA ou ProofWriter.
    - Conflict : les candidats donnent des labels différents.
    - Technical Unknown : absence de sortie solver, ou Unknown total sur FOLIO.
    """

    if not solver_outputs:
        return "Technical Unknown"

    clean_outputs = [normalize_label(output) for output in solver_outputs]
    unique_labels = set(clean_outputs)

    if len(unique_labels) == 1:
        label = next(iter(unique_labels))

        if label in ["True", "False"]:
            return "TF consensus"

        if label == "Unknown":
            if dataset == "FOLIO":
                return "Technical Unknown"

            return "Semantic Unknown"

    return "Conflict"


# ============================================================
# 10. RÉSUMÉ PAR FICHIER
# ============================================================

def summarize_file(dataset: str, model: str, path: Path) -> pd.DataFrame:
    data = read_json(path)
    results = get_results(data)

    rows = []

    for row in results:
        solver_outputs = get_solver_outputs(row)
        category = classify_instance(dataset, solver_outputs)

        rows.append(
            {
                "Dataset": dataset,
                "Model": model,
                "Category": category,
                "Correct": row_is_correct(row),
                "SolverOutputs": str(solver_outputs),
                "Source": str(path),
            }
        )

    instance_df = pd.DataFrame(rows)
    total = len(instance_df)

    summary_rows = []

    for category in CATEGORY_ORDER:
        subset = instance_df[instance_df["Category"] == category]
        count = len(subset)

        pct = 100.0 * count / total if total > 0 else 0.0
        acc = 100.0 * subset["Correct"].mean() if count > 0 else np.nan

        summary_rows.append(
            {
                "Dataset": dataset,
                "Model": model,
                "Category": category,
                "Count": count,
                "Pct": round(pct, 2),
                "Accuracy": round(acc, 2) if not np.isnan(acc) else np.nan,
                "Source": str(path),
            }
        )

    return pd.DataFrame(summary_rows)


def build_summary() -> Tuple[pd.DataFrame, pd.DataFrame]:
    summary_parts = []
    source_rows = []
    missing_rows = []

    for dataset in DATASET_ORDER:
        for model in MODEL_ORDER:
            path, candidates = find_poss_file(dataset, model)

            if path is None:
                missing_rows.append(
                    {
                        "Dataset": dataset,
                        "Model": model,
                        "Root": str(POSS_RESULTS_DIR),
                        "Top candidates": " | ".join(
                            [f"{score}: {candidate}" for score, candidate in candidates[:5]]
                        ),
                    }
                )
                continue

            summary = summarize_file(dataset, model, path)
            summary_parts.append(summary)

            source_rows.append(
                {
                    "Dataset": dataset,
                    "Model": model,
                    "Path": str(path),
                }
            )

    if missing_rows:
        missing_df = pd.DataFrame(missing_rows)

        print("\n" + "=" * 100)
        print("FICHIERS MANQUANTS")
        print("=" * 100)
        print(missing_df.to_string(index=False))

        raise FileNotFoundError(
            "\nCertains fichiers PBS/POSS n'ont pas été retrouvés automatiquement.\n"
            f"Diagnostic complet sauvegardé ici : {DIAGNOSTIC_CSV}\n"
            "Ouvre ce CSV pour voir les candidats détectés."
        )

    summary_df = pd.concat(summary_parts, ignore_index=True)
    source_df = pd.DataFrame(source_rows)

    return summary_df, source_df


# ============================================================
# 11. FIGURE 5
# ============================================================

def make_figure5(summary_df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.8), sharey=True)

    bar_width = 0.72
    x = np.arange(len(MODEL_ORDER))

    for ax, dataset in zip(axes, DATASET_ORDER):
        bottoms = np.zeros(len(MODEL_ORDER))

        for category in CATEGORY_ORDER:
            heights = []
            accuracies = []

            for model in MODEL_ORDER:
                row = summary_df[
                    (summary_df["Dataset"] == dataset)
                    & (summary_df["Model"] == model)
                    & (summary_df["Category"] == category)
                ]

                if row.empty:
                    heights.append(0.0)
                    accuracies.append(np.nan)
                else:
                    heights.append(float(row.iloc[0]["Pct"]))
                    accuracies.append(float(row.iloc[0]["Accuracy"]))

            bars = ax.bar(
                x,
                heights,
                bottom=bottoms,
                width=bar_width,
                color=CATEGORY_COLORS[category],
                edgecolor="white",
                linewidth=0.7,
                label=CATEGORY_LABELS[category],
            )

            for idx, bar in enumerate(bars):
                pct = heights[idx]
                acc = accuracies[idx]

                if pct <= 0:
                    continue

                xpos = bar.get_x() + bar.get_width() / 2

                if pct < MIN_EXTERNAL_LABEL_PCT:
                    continue

                if pct >= MIN_INTERNAL_LABEL_PCT:
                    ypos = bottoms[idx] + pct / 2

                    label = f"{np.ceil(pct):.0f}%"

                    if not np.isnan(acc):
                        label += f"\nacc={acc:.0f}%"

                    ax.text(
                        xpos,
                        ypos,
                        label,
                        ha="center",
                        va="center",
                        fontsize=7,
                        fontweight="bold",
                        color="white",
                    )

                else:
                    ypos_text = bottoms[idx] + pct + 5.0
                    ypos_line_bottom = bottoms[idx] + pct
                    ypos_line_top = ypos_text - 1.5

                    label = f"{pct:.0f}%"

                    if not np.isnan(acc):
                        label += f" | acc={acc:.0f}%"

                    ax.plot(
                        [xpos, xpos],
                        [ypos_line_bottom, ypos_line_top],
                        linestyle=":",
                        linewidth=1.0,
                        color="black",
                        alpha=0.7,
                    )

                    ax.text(
                        xpos,
                        ypos_text,
                        label,
                        ha="center",
                        va="bottom",
                        fontsize=7,
                        fontweight="bold",
                        color="black",
                    )

            bottoms += np.array(heights)

        ax.set_title(dataset, fontsize=12, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(MODEL_ORDER, fontsize=10)
        ax.grid(axis="y", linestyle="--", alpha=0.3)
        ax.set_ylim(0, 118)

    axes[0].set_ylabel("Instances (%)", fontsize=10)

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=4,
        frameon=False,
        bbox_to_anchor=(0.5, -0.02),
        fontsize=9,
    )

    plt.tight_layout(rect=[0.02, 0.10, 1, 1])

    fig.savefig(FIGURE_PNG, dpi=300, bbox_inches="tight")
    fig.savefig(FIGURE_PDF, bbox_inches="tight")

    plt.show()

    print("\nFigure 5 PNG :", FIGURE_PNG)
    print("Figure 5 PDF :", FIGURE_PDF)


# ============================================================
# 12. MAIN
# ============================================================

def main() -> None:
    if not POSS_RESULTS_DIR.exists():
        raise FileNotFoundError(
            f"Dossier POSS introuvable : {POSS_RESULTS_DIR}\n"
            "Vérifie que le dossier s'appelle bien POSS-LOGICLM."
        )

    diagnostic_df = build_diagnostic_report()
    diagnostic_df.to_csv(DIAGNOSTIC_CSV, index=False, encoding="utf-8-sig")

    print("=" * 100)
    print("DIAGNOSTIC DES FICHIERS FIGURE 5")
    print("=" * 100)
    print(
        diagnostic_df[
            ["Dataset", "Model", "Status", "Selected"]
        ].to_string(index=False)
    )

    summary_df, source_df = build_summary()

    summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")
    source_df.to_csv(SOURCE_CSV, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 100)
    print("SOURCES UTILISÉES POUR FIGURE 5")
    print("=" * 100)
    print(source_df.to_string(index=False))

    print("\n" + "=" * 100)
    print("RÉSUMÉ FIGURE 5")
    print("=" * 100)
    print(summary_df.to_string(index=False))

    make_figure5(summary_df)


if __name__ == "__main__":
    main()